# 16e · Submission — 4-model ensemble + aggregator (model 14) + aggregator (augmented model 14aug)
1. **Ensemble**: models 14, 15, 17, 17v2 on slices 6,8,…,18 → averaged (unchanged).
2. **Aggregator A** (19b): 5 FeatAgg models on model **14**'s slice features.
3. **Aggregator B** (19e): 5 FeatAgg models on model **14aug**'s slice features (Cristina's augmentation).
4. **Final = 0.5 · ensemble + 0.25 · aggregator A + 0.25 · aggregator B**, then averaged over planes.

**Settings:** Internet OFF · GPU · Inputs: competition data + outputs of `14`, `15`, `17_new_labels`, `17v2_labels`, `14aug`, `19b_featagg_honest`, `19e_featagg_aug`.

In [ ]:
# Cell 1: setup
import os, glob, numpy as np, pandas as pd, torch, timm, pydicom, cv2
from sklearn.metrics import roc_auc_score

BASE = '/kaggle/input/competitions/rsna-knee-abnormality-detection'
LABELS = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA',
          'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
PLANES = ['Sagittal', 'Coronal', 'Axial']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE, N_SLICES = 224, 24
TTA_SLICES = list(range(6, 19, 2))          # 6, 8, ..., 18

# (architecture, weights file name) for every model in the ensemble
MODELS = [
    ('resnet18', 'resnet18_random_slice_15ep.pt'),       # 14: old labels
    ('resnet34', 'resnet34_random_slice_15ep.pt'),       # 15: resnet34, old labels
    ('resnet18', 'resnet18_newlabels_v1_15ep.pt'),       # 17: Sherry's v1 labels
    ('resnet18', 'resnet18_newlabels_v2_15ep.pt'),       # 17v2: Sherry's v2 labels (Synovitis fix)
]
AGG_PATTERN = 'agg_B_featagg_seed*.pt'        # 5 aggregators from 19b (model 14 features)
AGG2_PATTERN = 'aggaug_B_featagg_seed*.pt'    # 5 aggregators from 19e (model 14aug features)
BACKBONE2 = ('resnet18', 'resnet18_aug_15ep.pt')  # 14aug: read by aggregator B only, not part of the ensemble
AGG_WEIGHT = 0.5                         # share of the final prediction that comes from the two aggregator sets (split equally)
LO, HI = 6, 19                           # the aggregator's baseline window (must match 19b)
print(DEVICE)

In [ ]:
# Cell 2: load every model (offline: pretrained=False, weights come from the .pt files)
models = []
for arch, name in MODELS:
    found = glob.glob(f'/kaggle/input/notebooks/**/{name}', recursive=True)
    assert len(found) == 1, f'{name}: expected 1 file, found {found}'
    m = timm.create_model(arch, pretrained=False, num_classes=12, in_chans=1)
    m.load_state_dict(torch.load(found[0], map_location=DEVICE))
    models.append(m.to(DEVICE).eval())
    print('loaded', arch, found[0])
print(len(models), 'models')

# --- the aggregator: identical class to notebook 19 (it must match exactly to load the weights)
import torch.nn as nn
class FeatAgg(nn.Module):
    def __init__(self, d=128, drop=0.3):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(512), nn.Dropout(drop), nn.Linear(512, d), nn.GELU())
        self.pos = nn.Parameter(torch.zeros(1, 24, d))
        layer = nn.TransformerEncoderLayer(d, nhead=4, dim_feedforward=2 * d, dropout=drop, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=1)
        self.query = nn.Linear(d, 12)
        self.out = nn.Linear(d, 12)
    def forward(self, feats, logits):
        h = self.enc(self.inp(feats) + self.pos)
        a = torch.softmax(self.query(h), dim=1)
        pooled = torch.einsum('bsf,bsd->bfd', a, h)
        corr = (self.out.weight.unsqueeze(0) * pooled).sum(-1) + self.out.bias
        return logits[:, LO:HI].mean(1) + corr

def load_aggs(pattern):
    files = sorted(glob.glob(f'/kaggle/input/notebooks/**/{pattern}', recursive=True))
    assert len(files) == 5, f'{pattern}: expected 5 files, found {files}'
    out = []
    for f in files:
        a = FeatAgg(); a.load_state_dict(torch.load(f, map_location=DEVICE)); out.append(a.to(DEVICE).eval())
    return out

aggs = load_aggs(AGG_PATTERN)            # read model 14's features
aggs2 = load_aggs(AGG2_PATTERN)          # read model 17v2's features
assert MODELS[0][1] == 'resnet18_random_slice_15ep.pt', 'models[0] must be model 14'
found2 = glob.glob(f'/kaggle/input/notebooks/**/{BACKBONE2[1]}', recursive=True)
assert len(found2) == 1, f'{BACKBONE2[1]}: expected 1 file, found {found2}'
backbone2 = timm.create_model(BACKBONE2[0], pretrained=False, num_classes=12, in_chans=1)
backbone2.load_state_dict(torch.load(found2[0], map_location=DEVICE))
backbone2 = backbone2.to(DEVICE).eval()
backbone = models[0]
print('aggregator B backbone:', found2[0])
print(len(aggs), '+', len(aggs2), 'aggregators loaded')

In [ ]:
# Cell 3: preprocessing, identical to training (02_preprocess)
def load_series(path):
    slices = [pydicom.dcmread(f) for f in glob.glob(f'{path}/*.dcm')]
    orient = np.array(slices[0].ImageOrientationPatient, float)
    normal = np.cross(orient[:3], orient[3:])
    slices.sort(key=lambda s: np.dot(normal, np.array(s.ImagePositionPatient, float)))
    return slices

def preprocess(path):
    vol = np.stack([cv2.resize(s.pixel_array.astype(np.float32), (IMG_SIZE, IMG_SIZE),
                               interpolation=cv2.INTER_AREA) for s in load_series(path)])
    vol = vol[np.linspace(0, len(vol) - 1, N_SLICES).round().astype(int)]
    lo, hi = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo) / (hi - lo + 1e-8), 0, 1)
    return (vol * 255).astype(np.uint8)                  # (24, 224, 224), same as the training .npy files

In [ ]:
# Cell 4: prediction for one volume and one knee
@torch.no_grad()
def agg_probs(bb, agg_list, x):
    """Backbone bb reads all 24 slices -> features -> average of the aggregators in agg_list."""
    with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):                 # same as notebook 18's extraction
        f = bb.forward_head(bb.forward_features(x), pre_logits=True)
        l = bb.get_classifier()(f)
    f, l = f.float().unsqueeze(0), l.float().unsqueeze(0)                    # (1,24,512), (1,24,12)
    return torch.stack([torch.sigmoid(a(f, l))[0] for a in agg_list]).mean(0)

@torch.no_grad()
def predict_volume(vol):
    """Returns (ensemble, aggregator A, aggregator B) probabilities, each (12,)."""
    x = torch.from_numpy((vol.astype(np.float32) / 255.0 - 0.45) / 0.225).unsqueeze(1).to(DEVICE)  # (24,1,224,224)
    ens = torch.stack([torch.sigmoid(m(x[TTA_SLICES])).mean(0) for m in models]).mean(0)
    a1 = agg_probs(backbone, aggs, x)
    a2 = agg_probs(backbone2, aggs2, x)
    return ens.cpu().numpy(), a1.cpu().numpy(), a2.cpu().numpy()

def n_files(path):
    return len(glob.glob(f'{path}/*.dcm'))

def predict_study(study, series_df, split):
    meta = series_df[series_df['StudyInstanceUID'] == study]
    folder = f'{BASE}/{split}_series/{study}'
    preds = []
    for plane in PLANES:
        cands = meta[(meta['Anatomical_Plane'] == plane) & (meta['Fluid_Sensitive'] == 1)]
        if len(cands) == 0:
            continue
        path = max([f'{folder}/{s}' for s in cands['SeriesInstanceUID']], key=n_files)   # most slices, like training
        try:
            preds.append(predict_volume(preprocess(path)))
        except Exception as e:
            print('failed:', study, plane, e)
    if not preds:                                         # fallback: no usable fluid-sensitive series, try any series
        for s in meta['SeriesInstanceUID']:
            try:
                preds.append(predict_volume(preprocess(f'{folder}/{s}')))
                break
            except Exception as e:
                print('fallback failed:', study, s, e)
    if not preds: return None
    return tuple(np.mean([p[k] for p in preds], 0) for k in range(3))   # average over planes

In [ ]:
# Cell 5: end-to-end check on the labelled train knees
train = pd.read_csv(f'{BASE}/train.csv')
train_series = pd.read_csv(f'{BASE}/train_series.csv')
lab = train.dropna(subset=LABELS).reset_index(drop=True)
out = [predict_study(s, train_series, 'train') for s in lab['StudyInstanceUID']]
E, A1, A2 = (np.stack([o[k] for o in out]) for k in range(3))
Y = lab[LABELS].values >= 0.5
def macro(P): return np.mean([roc_auc_score(Y[:, i], P[:, i]) for i in range(len(LABELS))])
blend = lambda w: (1 - w) * E + w * (A1 + A2) / 2
print(f'{len(lab)} labelled knees')
print(f'ensemble only             : {macro(E):.4f}   (16c: 0.7893)')
print(f'aggregator A (14) only    : {macro(A1):.4f}   (16c: 0.8033)')
print(f'aggregator B (14aug) only : {macro(A2):.4f}')
print(f'A+B averaged              : {macro((A1 + A2) / 2):.4f}')
print(f'16c recipe (ens + A, 50%) : {macro(0.5 * E + 0.5 * A1):.4f}   (16c: 0.8071)')
print(f'16e recipe (ens + A + B)  : {macro(blend(AGG_WEIGHT)):.4f}   <- used')

In [ ]:
# Cell 6: predict the test set and write submission.csv
sample = pd.read_csv(f'{BASE}/sample_submission.csv')
test_series = pd.read_csv(f'{BASE}/test_series.csv')

rows, n_fallback = [], 0
for i, study in enumerate(sample['StudyInstanceUID']):
    r = predict_study(study, test_series, 'test')
    p = None if r is None else (1 - AGG_WEIGHT) * r[0] + AGG_WEIGHT * (r[1] + r[2]) / 2
    if p is None:                                         # nothing usable at all -> neutral row, never a missing row
        p = np.full(len(LABELS), 0.5)
        n_fallback += 1
    rows.append([study, *p])
    if i % 100 == 0:
        print(i, '/', len(sample))

sub = pd.DataFrame(rows, columns=['StudyInstanceUID'] + LABELS)
assert list(sub.columns) == list(sample.columns), 'column mismatch'
assert len(sub) == len(sample) and sub['StudyInstanceUID'].equals(sample['StudyInstanceUID']), 'row mismatch'
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('fallback rows:', n_fallback)
sub.head()